# 4.3: Analogies: directions carry meaning

The idea is that the offset $`\mathbf{v}_b - \mathbf{v}_a`$ captures a *relation*, such as "female counterpart of," "capital city of," or "past tense of," and adding it to $`\mathbf{v}_c`$ applies the same relation to a new word. Mikolov et al. built a test set of such questions, semantic ones like country-capital pairs and syntactic ones like adjective-adverb or singular-plural pairs, and it became a standard way to evaluate word embeddings for several years.

In our toy space:

Earlier imports and definitions this listing relies on are included below so the notebook runs on its own.


In [ ]:
# Imports and definitions from earlier listings, included so this notebook runs alone.

# From A toy embedding space
import torch

import torch.nn.functional as F

words = ["king", "queen", "man", "woman", "prince", "apple", "banana"]

E = torch.tensor([
    [0.90,  0.75, 0.02, 0.70],   # king
    [0.88, -0.70, 0.05, 0.72],   # queen
    [0.08,  0.80, 0.01, 0.90],   # man
    [0.10, -0.78, 0.03, 0.88],   # woman
    [0.70,  0.70, 0.00, 0.55],   # prince
    [0.00,  0.02, 0.95, 0.05],   # apple
    [0.03, -0.01, 0.90, 0.02],   # banana
])

idx = {w: i for i, w in enumerate(words)}

def cosine(a, b):
    return (a @ b) / (a.norm() * b.norm())

print(f"cos(king, queen) = {cosine(E[idx['king']], E[idx['queen']]):.3f}")

print(f"cos(king, apple) = {cosine(E[idx['king']], E[idx['apple']]):.3f}")

# From Nearest neighbors
def nearest(query, E, k=3, exclude=()):
    sims = F.normalize(E, dim=1) @ F.normalize(query, dim=0)   # cosine with every row
    for i in exclude:
        sims[i] = -float("inf")                                  # never return these
    top = sims.topk(k)
    return [(words[i], round(s.item(), 3)) for s, i in zip(top.values, top.indices)]


In [ ]:
def analogy(a, b, c, E, k=1, exclude_inputs=True):
    """a is to b as c is to ?  Searches for the word nearest to b - a + c."""
    q = E[idx[b]] - E[idx[a]] + E[idx[c]]
    ex = [idx[a], idx[b], idx[c]] if exclude_inputs else []
    return nearest(q, E, k=k, exclude=ex)

print(analogy("man", "king", "woman", E))
print(analogy("man", "king", "woman", E, k=3, exclude_inputs=False))


**Expected output**

Output:

```text
[('queen', 0.996)]
[('queen', 0.996), ('woman', 0.802), ('king', 0.354)]
```
